In [1]:
import importlib
import parser as parser_module
importlib.reload(parser_module)

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from datetime import date

from parser import parse_factsheet, format_for_llm, extract_key_fields, extract_factsheet_date
from scraper import select_relevant_links, download_pdf, fetch_page_text
from enricher import enrich, format_enrichment_for_llm

load_dotenv(override=True)
openai = OpenAI()
MODEL = "gpt-4o"

In [5]:
dd_system_prompt = """
You are a senior investment analyst producing a due diligence brief for an institutional investor.
You will be given extracted text from a fund factsheet and supporting market data from yFinance.
Produce a structured brief in markdown. Be precise, use numbers where available, and avoid filler language.

CRITICAL RULES:
- Only use facts explicitly stated in the provided text. Do not infer or hallucinate any data.
- If a field is missing or unclear, write "Not disclosed" rather than guessing.
- Fund domicile and legal structure are NOT the same as portfolio geographic allocation.
- The benchmark listed in the factsheet may be "none" — state this honestly if so.
- The ## Market Context section contains benchmark proxy data from yFinance — use this 
  explicitly in the Performance section to compare fund returns against the proxy.
- For VaR, always include the confidence level, time horizon and exact percentage figure.
- When a fund has a stated absolute return target (e.g. "money market +2.5%"), 
  evaluate performance against that target first, not just vs the proxy ETF.
- VaR figures: ALWAYS use values from the ## Key Risk Figures (extracted) section, 
  never from raw text. The format "VaR 95 -10" means 10-day horizon, not the value.

Structure your output exactly as follows:

## Fund at a Glance
One paragraph: fund name, manager(s), AUM, domicile, inception date, stated benchmark 
(write "No benchmark" if none listed), SFDR classification.

## Investment Strategy
How the manager selects securities. Investment universe size, long/short approach if applicable,
key differentiators. Use exact quotes from the factsheet where helpful.

## Portfolio Characteristics
Investment style, actual portfolio geographic allocation (from factsheet charts/tables, 
NOT fund domicile), sector allocation with percentages, net equity exposure if stated.

## Performance
Fund returns across all available periods (YTD, 1Y, 3Y, 5Y, since inception).
Compare explicitly against the benchmark proxy from the Market Context section,
labelling it clearly as a proxy. If the fund has a stated return target, evaluate
performance against that target explicitly. Note max drawdown and return consistency across years.

## Risk Profile
Volatility p.a., Sharpe ratio, max drawdown, VaR 95 and VaR 99 with exact figures,
correlation to benchmark. Risk indicator rating (1-7 scale).

## Costs
TER (with date), management fee, performance fee with exact hurdle rate and 
high-watermark details. Entry/exit fees if any.

## Analyst Verdict
3-5 sentences. What type of investor or mandate this fund suits. Key strengths and weaknesses.
This brief is a first-pass screening tool designed to triage funds for deeper human review.
Make a decisive call based on available data:
- "Suitable" — fund has clear fit for institutional mandates, consistent track record, 
  reasonable costs, and meets or exceeds its stated return target
- "Requires Further Due Diligence" — genuinely ambiguous cases only: insufficient data, 
  unusual fee structure, inconsistent performance, or significant unexplained risks
- "Not Suitable" — clear misfit: excessive costs, poor risk-adjusted returns, or structural concerns
- A fund consistently meeting its stated return target with low costs, no performance fee, 
  and institutional-grade structure should be rated Suitable unless there is a specific 
  identified concern. Outperformance over 3Y vs proxy is a strong positive signal.

Recommendation: Suitable / Requires Further Due Diligence / Not Suitable — one-line rationale.
"""

In [3]:
def build_brief(fund_name: str, url: str, benchmark_hint: str = "", fund_ticker: str = ""):
    
    # step 1: scrape links
    print(f"Step 1: Scraping {url}...")
    docs = select_relevant_links(url)
    
    # step 2: find and download factsheet
    print("Step 2: Downloading factsheet...")
    factsheet_path = None
    for doc in docs.get("documents", []):
        if doc["type"] == "factsheet":
            factsheet_path = f"briefs/{fund_name.replace(' ', '_')}_factsheet.pdf"
            success = download_pdf(doc["url"], factsheet_path)
            if not success:
                factsheet_path = None
            break
    
    # step 3: parse factsheet
    print("Step 3: Parsing factsheet...")
    if factsheet_path:
        sections = parse_factsheet(factsheet_path)
        factsheet_text = format_for_llm(sections)
        factsheet_date = extract_factsheet_date(sections)
        print(f"  Factsheet date detected: {factsheet_date}")
    else:
        print("  No factsheet found, falling back to page text...")
        factsheet_text = fetch_page_text(url)
        factsheet_date = None

    # step 4: enrich with yfinance
    print("Step 4: Fetching market data...")
    enrichment = enrich(
        benchmark_hint=benchmark_hint,
        fund_ticker=fund_ticker,
        as_of_date=factsheet_date
    )
    enrichment_text = format_enrichment_for_llm(enrichment)
    
    # step 5: assemble prompt
    today = date.today().strftime("%d %B %Y")
    user_prompt = f"""
Fund Name: {fund_name}
Brief Date: {today}

{factsheet_text[:8000]}

{enrichment_text}
"""
    
    # step 6: stream the brief
    print("Step 5: Generating DD brief...\n")
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": dd_system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
    )

    full_response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        full_response += chunk.choices[0].delta.content or ""
        update_display(Markdown(full_response), display_id=display_handle.display_id)

    # step 7: save to file
    output_path = f"briefs/{fund_name.replace(' ', '_')}_{date.today().strftime('%Y%m%d')}.md"
    with open(output_path, "w") as f:
        f.write(f"# {fund_name} — Due Diligence Brief\n")
        f.write(f"*Generated: {today}*\n\n")
        f.write(full_response)
    print(f"\nBrief saved to {output_path}")

In [7]:
build_brief(
    fund_name="DWS Concept Kaldemorgen",
    url="https://funds.dws.com/en-ie/total-return-strategies/lu0599946893-dws-concept-kaldemorgen-lc/",
    benchmark_hint="msci world"
)

Step 1: Scraping https://funds.dws.com/en-ie/total-return-strategies/lu0599946893-dws-concept-kaldemorgen-lc/...
Step 2: Downloading factsheet...
Step 3: Parsing factsheet...
  No factsheet found, falling back to page text...
Step 4: Fetching market data...
Fetching benchmark data for: IWDA.AS anchored to latest
Step 5: Generating DD brief...



## Fund at a Glance
Fund name: DWS Concept Kaldemorgen  
Manager(s): Not disclosed  
AUM: Not disclosed  
Domicile: Not disclosed  
Inception date: Not disclosed  
Stated benchmark: No benchmark  
SFDR classification: Not disclosed  

## Investment Strategy
How the manager selects securities: Not disclosed  
Investment universe size, long/short approach if applicable, key differentiators: Not disclosed  

## Portfolio Characteristics
Investment style: Not disclosed  
Actual portfolio geographic allocation: Not disclosed  
Sector allocation with percentages: Not disclosed  
Net equity exposure: Not disclosed  

## Performance
Fund returns across available periods: Not disclosed  
- Comparison against Benchmark Proxy (IWDA.AS):
  - 1Y Performance: 25.8% (Benchmark Proxy)
  - 3Y Performance: 62.9% (Benchmark Proxy)

Since the fund does not have a specified return target or benchmark, performance evaluation is based solely on proxy comparison.

## Risk Profile
Volatility p.a.: Not disclosed  
Sharpe ratio: Not disclosed  
Max drawdown: Not disclosed  
VaR 95 and VaR 99 with exact figures: Not disclosed  
Correlation to benchmark: Not disclosed  
Risk indicator rating (1-7 scale): Not disclosed  

## Costs
TER (with date): Not disclosed  
Management fee: Not disclosed  
Performance fee with exact hurdle rate and high-watermark details: Not disclosed  
Entry/exit fees: Not disclosed  

## Analyst Verdict
Unfortunately, there is an insufficiency of disclosed information to make an informed recommendation. Critical data regarding performance, risk, and costs are missing, making it impossible to evaluate the fund's suitability accurately.

Recommendation: Requires Further Due Diligence — Insufficient data available to make a conclusive assessment.


Brief saved to briefs/DWS_Concept_Kaldemorgen_20260604.md


In [ ]:
build_brief(
    fund_name="Lupus alpha Smaller Euro Champions",
    url="https://www.lupusalpha.com/products/fund/lupus-alpha-smaller-euro-champions-a/",
    benchmark_hint="eurozone small"
)

Step 1: Scraping https://www.lupusalpha.com/products/fund/lupus-alpha-smaller-euro-champions-a/...
Found 76 total links, filtering with LLM...
Step 2: Downloading factsheet...
Downloaded: briefs/Lupus_alpha_Smaller_Euro_Champions_factsheet.pdf
Step 3: Parsing factsheet...
  Factsheet date detected: 29.05.2026
Step 4: Fetching market data...
Fetching benchmark data for: IEUS anchored to 29.05.2026
Step 5: Generating DD brief...



## Fund at a Glance
- **Fund Name:** Lupus alpha Smaller Euro Champions
- **Manager(s):** Marcus Ratz, Franz Führer
- **AUM:** 61.99 million EUR
- **Domicile:** Luxembourg
- **Inception Date:** 22.10.2001
- **Stated Benchmark:** Euro Stoxx TMI Small Net Return
- **SFDR Classification:** 8

## Investment Strategy
The fund invests in promising small- and medium-sized companies within Euroland, emphasizing market leaders in niche areas with significant market share. The strategy employs a "consistent bottom-up approach," focusing on "quality titles" that provide potential for high value increases due to the inefficient information environment in the small and mid-cap sector. Ethical and ESG criteria are integrated into the investment process.

## Portfolio Characteristics
- **Investment Style:** Eurozone small and mid caps / Quality
- **Geographic Allocation:** Eurozone-focused
- **Sector Allocation:** Not disclosed in detail
- **Net Equity Exposure:** Investment ratio at 97.93%

## Performance
- **YTD Return:** 9.36% (Benchmark Proxy: 16.98%) 
- **1 Year Return:** 12.41% (Benchmark Proxy: 16.98%)
- **3 Year Return:** 22.26% (Benchmark Proxy: 54.62%)
- **5 Year Return:** 13.58% (Benchmark Performance Disclosed)
- **Since Inception p.a.:** 8.24% (Benchmark: 8.22%)
- **Comparison to Proxy:** Underperformed IEUS (1Y: 16.98%, 3Y: 54.62%).
- **Max Drawdown:** -61.83%
- **Return Consistency:** Performance varied significantly. Notable volatilities observed, with -21.47% in 2022 and higher returns like 23.45% in 2017.

## Risk Profile
- **Volatility p.a.:** 17.91%
- **Sharpe Ratio:** 0.39
- **Max Drawdown:** -61.83%
- **VaR 95:** Not disclosed
- **VaR 99:** Not disclosed
- **Correlation to Benchmark:** Not disclosed
- **Risk Indicator Rating:** 4 (Moderately fluctuating unit price)

## Costs
- **TER:** 2.08% p.a. (as of 31.12.2025)
- **Management Fee:** 1.50%
- **Performance Fee:** 17.5% of outperformance
- **Entry/Exit Fees:** Max initial charge up to 5%

## Analyst Verdict
**Recommendation: Not Suitable** — While the fund provides exposure to Eurozone small and mid caps with ESG integration, it underperforms against both its stated benchmark and the market proxy across multiple periods. High volatility and a significant max drawdown are concerns, and combined with the relatively high TER and performance fees, the fund is not deemed suitable for institutional investors seeking strong risk-adjusted performance relative to benchmarks.


Brief saved to briefs/Lupus_alpha_Smaller_Euro_Champions_20260604.md
